# Vast.ai Training Plan — Baseline RNN + 5‑gram LM (Kaggle Brain‑to‑Text '25)

This notebook guides you through running the full pipeline on a Vast.ai instance, including the 5‑gram n‑gram language model (requires ~300GB RAM) and optional OPT‑6.7b rescoring.

Prereqs (choose during Vast.ai instance creation)
- Ubuntu 22.04 image (or compatible)
- NVIDIA driver + CUDA support (provider image typically has it)
- GPU with ≥16GB VRAM (A100 40/80GB preferred if doing OPT rescoring)
- System RAM ≥ 300GB for the 5‑gram LM
- Disk ≥ 200GB (data + models + cache)

Outcomes
- Trained baseline GRU decoder
- Validation WER
- Test CSV for Kaggle submission



## 0) Connect to the Vast.ai instance (from your local machine)

Replace placeholders with your instance's IP and SSH key.

```bash
# On your local computer terminal (not inside the instance):
    ssh -i /path/to/your_key.pem root@<INSTANCE_PUBLIC_IP>
```

If you prefer a non-root user, create one after login:

```bash
adduser ubuntu
usermod -aG sudo ubuntu
su - ubuntu
```


In [ ]:
# 1) System prep (inside the Vast.ai instance)
set -euxo pipefail

# Update and install system dependencies
sudo apt-get update -y
sudo apt-get install -y build-essential cmake git curl wget redis-server

# (Optional) Stop auto-start; we'll run redis manually
sudo systemctl disable redis-server || true

# Install Miniconda (per-user)
cd ~
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p $HOME/miniconda3
source ~/miniconda3/bin/activate

# Ensure conda is on PATH in future sessions
echo 'source ~/miniconda3/bin/activate' >> ~/.bashrc



In [ ]:
# 2) Get the repository and set up environments
set -euxo pipefail

# Choose a workspace
mkdir -p ~/work
cd ~/work

# If you already uploaded the project, skip cloning and cd into it
# Otherwise clone from your remote (replace with your repo URL if needed)
if [ ! -d nejm-brain-to-text ]; then
  git clone https://github.com/<your-user-or-org>/nejm-brain-to-text.git
fi
cd nejm-brain-to-text

# Create the training env (b2txt25)
./setup.sh
# Verify; then activate
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25
python -c "import torch; print('Torch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())"

# Create the LM env (b2txt25_lm) for n-gram + OPT rescoring
conda deactivate || true
./setup_lm.sh
conda activate b2txt25_lm
python -c "import torch; print('Torch(LM):', torch.__version__)"

# Back to training env for data download
conda activate b2txt25



In [ ]:
# 3) Download data from Dryad
set -euxo pipefail
cd ~/work/nejm-brain-to-text

conda activate b2txt25
python download_data.py

# Quick verify
ls -R data | head -n 80



## 4) Prepare the 5‑gram language model files

The 5‑gram LM requires ~300GB RAM. You need the pretrained 5‑gram model from the paper’s Dryad link (separate from the main dataset): `languageModel_5gram.tar.gz`.

Steps:
- Download `languageModel_5gram.tar.gz` to the instance (or upload via scp).
- Extract to `language_model/pretrained_language_models/openwebtext_5gram_lm_sil`.

If you also need the 3‑gram: `languageModel.tar.gz` to `openwebtext_3gram_lm_sil`.



In [ ]:
# 4a) (Example) Place and extract 5‑gram LM tarball
set -euxo pipefail
cd ~/work/nejm-brain-to-text

# Adjust the source URL/path as appropriate. If uploaded via scp, skip wget.
# wget -O languageModel_5gram.tar.gz "<DIRECT_DOWNLOAD_URL_TO_LANGUAGEMODEL_5GRAM_TARBALL>"

mkdir -p language_model/pretrained_language_models/openwebtext_5gram_lm_sil
# tar -xzf languageModel_5gram.tar.gz -C language_model/pretrained_language_models/openwebtext_5gram_lm_sil

# List files if already extracted
ls -la language_model/pretrained_language_models/openwebtext_5gram_lm_sil || true



In [ ]:
# 5) Build/verify LM runtime (already handled by setup_lm.sh)
set -euxo pipefail
cd ~/work/nejm-brain-to-text

# Ensure LM env is active
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25_lm

# If you need to rebuild parts, see language_model/README.md
# For now, just verify language-model-standalone.py is runnable:
python -c "import sys; import os; print('LM script exists:', os.path.exists('language_model/language-model-standalone.py'))"



In [ ]:
# 6) Start Redis and the 5‑gram LM (in screen/tmux or a second shell)
set -euxo pipefail
cd ~/work/nejm-brain-to-text

# Start redis
sudo redis-server --daemonize yes
redis-cli ping || true

# Activate LM env
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25_lm

# Start the 5‑gram LM with OPT rescoring (requires large RAM + GPU VRAM)
# If you hit memory pressure, try removing --do_opt to disable OPT rescoring.
python language_model/language-model-standalone.py \
  --lm_path language_model/pretrained_language_models/openwebtext_5gram_lm_sil \
  --rescore --do_opt --nbest 100 \
  --acoustic_scale 0.325 --blank_penalty 90 --alpha 0.55 \
  --redis_ip localhost --gpu_number 0



In [ ]:
# 7) Train the baseline RNN (run in another shell or after LM is up)
set -euxo pipefail
cd ~/work/nejm-brain-to-text/model_training

source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25

# Optionally adjust rnn_args.yaml here (GPU index, num_training_batches, etc.)
python - << 'PY'
from omegaconf import OmegaConf
args = OmegaConf.load('rnn_args.yaml')
args.gpu_number = '0'
# args.num_training_batches = 120000   # uncomment to change
args.output_dir = 'trained_models/baseline_rnn'
args.checkpoint_dir = 'trained_models/baseline_rnn/checkpoint'
OmegaConf.save(config=args, f='rnn_args.yaml')
print('Saved updated rnn_args.yaml')
PY

# Start training
python train_model.py



In [ ]:
# 8) Evaluate on validation (compute WER)
set -euxo pipefail
cd ~/work/nejm-brain-to-text/model_training

source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25

# Ensure the 5-gram LM is running and connected to redis before this step
python evaluate_model.py \
  --model_path trained_models/baseline_rnn \
  --data_dir ../data/hdf5_data_final \
  --eval_type val \
  --gpu_number 1



In [ ]:
# 9) Evaluate on test (produce submission CSV)
set -euxo pipefail
cd ~/work/nejm-brain-to-text/model_training

source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate b2txt25

python evaluate_model.py \
  --model_path trained_models/baseline_rnn \
  --data_dir ../data/hdf5_data_final \
  --eval_type test \
  --gpu_number 1

# CSV saved under the model path, e.g.:
# trained_models/baseline_rnn/baseline_rnn_test_predicted_sentences_YYYYMMDD_HHMMSS.csv



## Notes & Troubleshooting

- Memory for 5‑gram LM
  - Ensure your instance has ≥300GB RAM. If not, use 3‑gram (≈60GB RAM) or 1‑gram.
- VRAM for OPT 6.7b
  - Requires ~12.4GB VRAM to load. Remove `--do_opt` to disable rescoring if VRAM is insufficient.
- Check LM connectivity
  - `redis-cli ping` returns `PONG`.
  - LM console should print a successful Redis connection.
- Resume training
  - Set `init_from_checkpoint: true` and `init_checkpoint_path` in `rnn_args.yaml` if you want to continue from a previous run.
- Security
  - Stop the instance when finished to avoid charges.

